In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ===== 입력값 =====
csv_path = Path(r"D:\uwb-config\calibration\data\64_uwb_20260710_184516_631_KST.csv")
current_ant_delay = 16450   # 현재 TX/RX 안테나 지연 상수
target_distance_m = 8.0
trim_seconds = 5
anchor_id = 6

# 같은 상수를 어디까지 같이 바꿀지
# 4: tag TX/RX + anchor TX/RX 모두 같은 tick만큼 보정
# 2: 한쪽 보드의 TX/RX만 보정
adjusted_delay_registers = 4

# DW3000/DW1000 device time unit
DWT_TIME_UNITS = 1.0 / (499.2e6 * 128.0)
SPEED_OF_LIGHT = 299_702_547.0
M_PER_DTU = DWT_TIME_UNITS * SPEED_OF_LIGHT

df = pd.read_csv(csv_path)
total_packets = len(df)

if anchor_id is not None:
    df = df[df["ANC"] == anchor_id].copy()
    if df.empty:
        raise ValueError(f"ANC == {anchor_id}인 데이터가 없습니다.")
filtered_packets = len(df)

df["Timestamp_dt"] = pd.to_datetime(df["Timestamp"], format="%H:%M:%S.%f")

# 자정 넘어가는 로그 보정
day_offset = (df["Timestamp_dt"].diff().dt.total_seconds() < 0).cumsum()
df["Timestamp_dt"] = df["Timestamp_dt"] + pd.to_timedelta(day_offset, unit="D")

start_t = df["Timestamp_dt"].iloc[0]
end_t = df["Timestamp_dt"].iloc[-1]

mid = df[
    (df["Timestamp_dt"] >= start_t + pd.Timedelta(seconds=trim_seconds)) &
    (df["Timestamp_dt"] <= end_t - pd.Timedelta(seconds=trim_seconds))
].copy()

mean_dist = mid["Dist"].mean()
std_dist = mid["Dist"].std()
error_m = target_distance_m - mean_dist

# 측정값이 짧으면 delay를 낮추는 방향.
# DS-TWR에서 TX/RX delay를 양쪽 보드 모두 같이 움직이면 range 변화량은
# 대략 -2 * tick * M_PER_DTU.
range_per_tick = -(adjusted_delay_registers / 2.0) * M_PER_DTU
delta_tick = error_m / range_per_tick
suggested_ant_delay = int(round(current_ant_delay + delta_tick))

print(f"파일: {csv_path}")
print(f"전체 패킷 수: {total_packets}")
if anchor_id is not None:
    print(f"앵커 필터: ANC == {anchor_id}")
    print(f"필터 후 패킷 수: {filtered_packets}")
print(f"분석 구간: {mid['Timestamp'].iloc[0]} ~ {mid['Timestamp'].iloc[-1]}")
print(f"중간 패킷 수: {len(mid)}")
print(f"중간 패킷 번호: {mid['#Packet'].iloc[0]} ~ {mid['#Packet'].iloc[-1]}")
print()
print(f"현재 평균 거리: {mean_dist:.6f} m")
print(f"표준편차: {std_dist:.6f} m")
print(f"목표 거리: {target_distance_m:.6f} m")
print(f"오차: {error_m:+.6f} m")
print()
print(f"현재 안테나 지연 상수: {current_ant_delay}")
print(f"추천 보정량: {delta_tick:+.2f} tick")
print(f"추천 안테나 지연 상수: {suggested_ant_delay}")

파일: D:\uwb-config\calibration\data\63_uwb_20260710_183939_481_KST.csv
전체 패킷 수: 367
앵커 필터: ANC == 6
필터 후 패킷 수: 367
분석 구간: 18:39:44.621 ~ 18:41:45.618
중간 패킷 수: 339
중간 패킷 번호: 15587 ~ 15925

현재 평균 거리: 7.885059 m
표준편차: 0.018880 m
목표 거리: 8.000000 m
오차: +0.114941 m

현재 안테나 지연 상수: 16445
추천 보정량: -12.25 tick
추천 안테나 지연 상수: 16433
